# FashionMNIST MLP classifier (PyTorch)

Scripts `01_setup.py` through `09_optuna.py` combined in execution order. Each cell depends on variables defined by earlier cells, so run top to bottom.

Requirements: `torch`, `torchvision`, `torchmetrics`, `matplotlib`, `numpy`, `optuna`.

## `01_setup.py`

Setup: imports and device selection

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
import torchvision
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | torchmetrics {torchmetrics.__version__}")
print(f"Using device: {device}")

## `02_data.py`

Data: load FashionMNIST, train/val split, DataLoaders

In [ ]:
import torchvision.transforms.v2 as T

transform = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_full = datasets.FashionMNIST(root="data", train=True, download=True, transform=transform)
test_ds = datasets.FashionMNIST(root="data", train=False, download=True, transform=transform)
class_names = train_full.classes

train_ds, val_ds = random_split(train_full, [55_000, 5_000], generator=torch.Generator().manual_seed(SEED))

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"train: {len(train_ds)} | val: {len(val_ds)} | test: {len(test_ds)}")

image, label = train_ds[0]
print(f"Sample shape: {tuple(image.shape)}, dtype: {image.dtype}, range: [{image.min():.2f}, {image.max():.2f}]")
print(f"Sample label: {label} ({class_names[label]})")

## `03_model.py`

Model: MLP classifier and loss

In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs=28 * 28, n_hidden1=300, n_hidden2=100, n_classes=10):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes),
        )

    def forward(self, X):
        return self.mlp(X)


model = ImageClassifier(n_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()

print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## `04_train.py`

Training: evaluation with torchmetrics and training loop

In [ ]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()


def train2(model, optimizer, loss_fn, metric, train_loader, valid_loader, n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        model.train()
        metric.reset()
        total_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            metric.update(y_pred, y_batch)

        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(evaluate_tm(model, valid_loader, metric).item())
        print(
            f"Epoch {epoch + 1}/{n_epochs}, "
            f"train loss: {history['train_losses'][-1]:.4f}, "
            f"train accuracy: {history['train_metrics'][-1]:.2%}, "
            f"valid accuracy: {history['valid_metrics'][-1]:.2%}"
        )
    return history


LEARNING_RATE = 0.1
N_EPOCHS = 20

optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=len(class_names)).to(device)

history = train2(model, optimizer, criterion, accuracy, train_loader, val_loader, N_EPOCHS)

## `05_plot_accuracy.py`

Plot: training and validation accuracy per epoch

In [ ]:
epochs = range(1, len(history["train_metrics"]) + 1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs, history["train_metrics"], marker="o", label="Training accuracy")
ax.plot(epochs, history["valid_metrics"], marker="s", label="Validation accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("FashionMNIST MLP: accuracy per epoch")
ax.set_xticks(list(epochs))
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## `06_predict.py`

Predict: 3 validation images, class names, softmax probabilities, top-4

In [ ]:
X_new, y_new = next(iter(val_loader))
X_new, y_new = X_new[:3].to(device), y_new[:3].to(device)

model.eval()
with torch.no_grad():
    y_logits = model(X_new)

y_pred = y_logits.argmax(dim=1)
print("Predicted:", [class_names[i] for i in y_pred])
print("True:     ", [class_names[i] for i in y_new])
print("Correct:  ", (y_pred == y_new).tolist())

y_proba = F.softmax(y_logits, dim=1)
print("\nSoftmax probabilities (rounded):")
print(y_proba.round(decimals=3).cpu())

top_k = torch.topk(y_proba, k=4, dim=1)
for i, (probas, indices) in enumerate(zip(top_k.values, top_k.indices)):
    top4 = ", ".join(f"{class_names[idx]} {p:.1%}" for p, idx in zip(probas, indices))
    print(f"Image {i} (true: {class_names[y_new[i]]}) top-4: {top4}")

## `07_test_eval.py`

Test evaluation: accuracy on the held-out test set

In [ ]:
test_accuracy = evaluate_tm(model, test_loader, accuracy).item()
print(f"Validation accuracy (last epoch): {history['valid_metrics'][-1]:.2%}")
print(f"Test accuracy:                   {test_accuracy:.2%}")

## `08_save_load.py`

Save/load: state_dict with hyperparameters, verify predictions match

In [ ]:
from pathlib import Path

model_hparams = {"n_inputs": 28 * 28, "n_hidden1": 300, "n_hidden2": 100, "n_classes": len(class_names)}
train_hparams = {"lr": LEARNING_RATE, "n_epochs": N_EPOCHS, "batch_size": BATCH_SIZE, "seed": SEED}

checkpoint_path = Path("models/fashion_mnist_mlp.pt")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_hparams": model_hparams,
        "train_hparams": train_hparams,
        "class_names": class_names,
    },
    checkpoint_path,
)
print(f"Saved checkpoint to {checkpoint_path}")

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
loaded_model = ImageClassifier(**checkpoint["model_hparams"]).to(device)
loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model.eval()
print(f"Loaded model with {checkpoint['model_hparams']} (trained with {checkpoint['train_hparams']})")

model.eval()
with torch.no_grad():
    original_logits = model(X_new)
    loaded_logits = loaded_model(X_new)

assert torch.allclose(original_logits, loaded_logits), "Loaded model logits differ from the original"
assert torch.equal(original_logits.argmax(dim=1), loaded_logits.argmax(dim=1))
print("Predictions match:", [checkpoint["class_names"][i] for i in loaded_logits.argmax(dim=1)])

## `09_optuna.py`

Hyperparameter tuning: lr and n_hidden with Optuna

In [ ]:
import optuna

N_TRIALS = 5
N_TRIAL_EPOCHS = 10


def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)

    torch.manual_seed(SEED)
    trial_model = ImageClassifier(n_hidden1=n_hidden, n_hidden2=n_hidden, n_classes=len(class_names)).to(device)
    trial_optimizer = torch.optim.SGD(trial_model.parameters(), lr=lr)
    trial_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=len(class_names)).to(device)

    trial_history = train2(
        trial_model, trial_optimizer, criterion, trial_accuracy, train_loader, val_loader, N_TRIAL_EPOCHS
    )
    return trial_history["valid_metrics"][-1]


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS)

print("\nAll trials:")
for t in study.trials:
    print(f"  Trial {t.number}: lr={t.params['lr']:.2e}, n_hidden={t.params['n_hidden']}, valid accuracy={t.value:.2%}")
print(f"Best params: {study.best_params}")
print(f"Best valid accuracy: {study.best_value:.2%}")